# 01 — Configuration & Calibration Store

Two objects manage the state of a qubit over a lab session:

| Class | Role |
|---|---|
| `ExperimentConfig` | Live, mutable config dict per qubit (in-memory) |
| `CalibrationStore` | Timestamped JSON store — survives process restarts |

They are kept in sync: every time an experiment finds a better parameter value,
it calls `config_all.update(...)` *and* `store.set(...)` in one step.

In [1]:
import sys; sys.path.insert(0, '../')
from QickworkspaceV2 import ExperimentConfig, CalibrationStore

## ExperimentConfig

`config_list` is a list of nested dicts (one per qubit).  Sub-dicts (`ch`, `res`, `qb`, `cooling`)
are automatically **flattened** so every experiment sees a single flat dict.

In [2]:
config_list = [
    {
        "name": "Q1",
        "ch":  {"ro_ch": 0, "res_ch": 0, "qb_ch": 1},
        "res": {"res_freq_ge": 6700.0, "res_gain": 0.5},
        "qb":  {"qb_freq_ge": 5000.0, "pi_gain_ge": 0.5, "sigma": 0.025},
        "reps": 100, "relax_delay": 300.0, "steps": 101,
    },
    {
        "name": "Q2",
        "ch":  {"ro_ch": 1, "res_ch": 2, "qb_ch": 3},
        "res": {"res_freq_ge": 6850.0, "res_gain": 0.5},
        "qb":  {"qb_freq_ge": 5150.0, "pi_gain_ge": 0.48, "sigma": 0.025},
        "reps": 100, "relax_delay": 300.0, "steps": 101,
    },
]

cfg_all = ExperimentConfig(config_list)
print(cfg_all)
print('Qubits:', cfg_all.qubit_names())

ExperimentConfig(qubits=['Q1', 'Q2'])
Qubits: ['Q1', 'Q2']


In [3]:
# get_qubit returns a COPY — safe to mutate
cfg = cfg_all.get_qubit('Q1')
print('qb_freq_ge:', cfg['qb_freq_ge'])

# also accessible by index
cfg2 = cfg_all.get_qubit(1)   # Q2
print('Q2 res_freq_ge:', cfg2['res_freq_ge'])

qb_freq_ge: 5000.0
Q2 res_freq_ge: 6850.0


In [4]:
# Update a parameter in-place (mutates the live config)
cfg_all.update('qb_freq_ge', 4998.7, q_index='Q1')

# Batch update
cfg_all.update([('pi_gain_ge', 0.51), ('sigma', 0.026)], q_index='Q1')

print('Updated qb_freq_ge:', cfg_all.get_qubit('Q1')['qb_freq_ge'])
print('Updated pi_gain_ge:', cfg_all.get_qubit('Q1')['pi_gain_ge'])

Updated qb_freq_ge: 4998.7
Updated pi_gain_ge: 0.51


## CalibrationStore

Persists calibration parameters as timestamped JSON.  Enables **staleness detection** — 
if a parameter was written more than `max_age_hours` ago, `is_stale()` returns `True`
and you know it's time to re-calibrate.

In [5]:
import tempfile, os

# Use a temp file so this notebook is self-contained
store_path = os.path.join(tempfile.gettempdir(), 'demo_cal_store.json')
store = CalibrationStore(store_path, default_max_age_hours=24)
print(store)

CalibrationStore(path='C:\\Users\\QEL\\AppData\\Local\\Temp\\demo_cal_store.json', qubits=['Q1'], entries=5)


In [6]:
# Write parameters
store.set('Q1', 'qb_freq_ge',  4998.7)
store.set('Q1', 'pi_gain_ge',  0.51)
store.set('Q1', 'res_freq_ge', 6701.3)
store.set('Q1', 'T1_us',       45.2)
store.set('Q1', 'T2r_us',      18.6)

# Read them back
print('qb_freq_ge :', store.get('Q1', 'qb_freq_ge'))
print('timestamp  :', store.timestamp('Q1', 'qb_freq_ge'))
print('stale?     :', store.is_stale('Q1', 'qb_freq_ge'))

qb_freq_ge : 4998.7
timestamp  : 2026-05-04 00:18:29.367958
stale?     : False


In [7]:
# Batch update from a dict (e.g., from an experiment result)
new_params = {'qb_freq_ge': 4999.0, 'pi_gain_ge': 0.502}
store.update_from_dict('Q1', new_params)

# Human-readable summary
print(store.summary('Q1'))

[Q1]
  qb_freq_ge                   = 4999.0  @ 2026-05-04T00:18:29.382484
  pi_gain_ge                   = 0.502  @ 2026-05-04T00:18:29.382484
  res_freq_ge                  = 6701.3  @ 2026-05-04T00:18:29.368958
  T1_us                        = 45.2  @ 2026-05-04T00:18:29.369959
  T2r_us                       = 18.6  @ 2026-05-04T00:18:29.370470


In [8]:
# Flat dict — handy for feeding back into ExperimentConfig
flat = store.to_flat_dict('Q1')
print('Stored values:', flat)

# Sync store → live config (e.g., on session startup)
for key, val in flat.items():
    if key in cfg_all.get_qubit('Q1'):
        cfg_all.update(key, val, q_index='Q1')

print('Live qb_freq_ge after sync:', cfg_all.get_qubit('Q1')['qb_freq_ge'])

Stored values: {'qb_freq_ge': 4999.0, 'pi_gain_ge': 0.502, 'res_freq_ge': 6701.3, 'T1_us': 45.2, 'T2r_us': 18.6}
Live qb_freq_ge after sync: 4999.0


## Check staleness before a session

A typical session-start pattern: decide which experiments need to run based on how old their
last calibration is.

In [9]:
params_to_check = [
    ('res_freq_ge', 48),   # re-calibrate if older than 48 h
    ('qb_freq_ge',  12),   # re-calibrate if older than 12 h
    ('pi_gain_ge',  12),
    ('T1_us',       24),
]

print('Stale parameters for Q1:')
for param, max_age in params_to_check:
    stale = store.is_stale('Q1', param, max_age_hours=max_age)
    tag = '*** STALE ***' if stale else 'OK'
    print(f'  {param:<20s} {tag}')

Stale parameters for Q1:
  res_freq_ge          OK
  qb_freq_ge           OK
  pi_gain_ge           OK
  T1_us                OK


**Next:** [02_running_experiments.ipynb](02_running_experiments.ipynb) — running the standard experiment sequence.